# Programs and optimizers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/07_program.ipynb)

`MathematicalProgram` and `Optimizer`: the solver boundary, from tuning a controller on its problem to a gradient through a solve.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. Tune a controller: the evaluation becomes a program

`pe.program(wrt)` turns a problem evaluator into a `MathematicalProgram`: the decision
vector $z$ is the leaves of `pe.params` named by `wrt` (dotted paths such as
`"controller.Kp"`), the objective is the evaluation's `objective`, and the problem's
declared sets become constraint rows $g(z) \ge 0$ (here the margin to `Xf`). On JAX the
program carries its exact gradient. `Optimizer(program, method="auto").solve()` solves it,
and `pe.unpack` writes the solution back into the tree.

The task is the one of `06_compile_and_evaluate`: a digital PD law bringing an inverted
pendulum back up. Its LQR benchmark scored $J \approx 2.26$.

In [ ]:
import numpy as np

from minilink import (
    PID,
    BallSet,
    InvertedPendulum,
    Optimizer,
    PlanningProblem,
    QuadraticCost,
)

plant = InvertedPendulum()
cost = QuadraticCost.from_system(plant, Q=np.diag([10.0, 1.0]), R=0.1 * np.eye(1))
problem = PlanningProblem(
    plant,
    cost=cost,
    x_start=np.array([0.4, 0.0]),
    tf=4.0,
    Xf=BallSet(np.zeros(2), 0.05),
)
pe = problem.compile(PID(Kp=20.0, Kd=4.0) % 0.01)

wrt = ["controller.Kp", "controller.Kd"]
program = pe.program(wrt)
print(program)

In [ ]:
best = Optimizer(program, method="auto").solve()
print(best)

tuned = pe.unpack(best.z, wrt)
print(pe.evaluate(tuned))

In [ ]:
pe.loop.plot_trajectory(pe.trajectory(tuned))

`pack` and `unpack` are the two directions between the tree and the vector: `pack(params,
wrt)` reads the named leaves into one flat $z$ (the program's start is
`pack(pe.params, wrt)`), and `unpack(z, wrt, params=)` returns a copy of the tree with those
leaves replaced.

In [ ]:
pe.pack(pe.params, wrt), pe.pack(tuned, wrt)

## 2. `MathematicalProgram`: the one NLP shape

Every transcription in minilink, a tuning program, a trajectory optimization, an MPC tick,
lowers to the same textbook form:

$$\min_z J(z) \quad \text{s.t.} \quad h(z) = 0,\quad g(z) \ge 0,\quad
\text{lower} \le z \le \text{upper}.$$

A program is a description: pure callables of $z$ and the bounds, no solver state. Written
by hand, with `backend="jax"` so the solver gets exact derivatives:

In [ ]:
import jax.numpy as jnp

from minilink import MathematicalProgram


def J(z):
    return 0.5 * z @ z


def h(z):
    return jnp.array([z[0] + z[1] - 1.0])


def g(z):
    return jnp.array([z[0] - 0.7])


nlp = MathematicalProgram(
    n_z=2, J=J, h=h, g=g, lower=[0.0, 0.0], upper=[1.0, 1.0], backend="jax"
)
solution = Optimizer(nlp, z0=np.array([0.5, 0.5]), method="auto").solve()
solution.z, solution.success, solution.cost

`Optimizer(program, z0, method=)` binds a program to a solver. The methods, named by
algorithm:

| `method` | Algorithm | Admits |
| --- | --- | --- |
| `"auto"` | L-BFGS when unconstrained or box-bounded, SLSQP otherwise | any program |
| `"l-bfgs"` | quasi-Newton | bounds only |
| `"scipy_slsqp"`, `"scipy_trust_constr"` | sequential quadratic programming, trust region | constraints |
| `"ipopt"` | interior point (needs `cyipopt`) | constraints |
| `"gd"`, `"adam"` | first-order, projected on the box; `iterations=` fixes the step count | bounds only |

The constructor's default is `"scipy_slsqp"`, so pass `method="auto"` to get the class
default. First-order methods take their step size through `options={"learning_rate": ...}`.

## 3. Parametric programs

A program may declare a parameter tree. Its callables then take `(z, params)`, and
`solve(params=)` solves for new values without recompiling: the shape an MPC re-solves at
every tick, with the measured state as a parameter. A least-distance program to a target
$\bar z$:

In [ ]:
def J_target(z, params):
    z_bar = params["z_bar"]

    e = z - z_bar
    J = 0.5 * e @ e

    return J


target_program = MathematicalProgram(
    n_z=2, J=J_target, params={"z_bar": np.array([1.0, -0.5])}, backend="jax"
)
optimizer = Optimizer(target_program, z0=np.zeros(2), method="l-bfgs")
optimizer.solve().z, optimizer.solve(params={"z_bar": np.array([3.0, 2.0])}).z

## 4. A fixed-iteration solve is a system, and it is differentiable

With `method="gd"` or `"adam"` and `iterations=K`, the solve is exactly $K$ steps of
$z_{k+1} = z_k - \eta \nabla J(z_k; p)$, one `lax.scan` on JAX. `as_step_system()` shows it
as what it is, a `StepSystem` whose state is the iterate: `compute_rollout(K)` gives the
iterates.

In [ ]:
descent = Optimizer(
    target_program,
    z0=np.zeros(2),
    method="gd",
    options={"learning_rate": 0.3},
    iterations=20,
)
iterations = descent.as_step_system()
iterates = iterations.compute_rollout(20)

In [ ]:
iterations.plot_rollout()

The fixed-count solve is a pure function of its parameters, so `jax.grad` and
`jax.jacobian` differentiate through it: here $\partial z_K / \partial \bar z$, which tends
to the identity as the descent converges ($1 - 0.7^{20}$ after twenty steps). This is how a
solver sits inside a larger differentiable pipeline, such as an MPC tuned by gradient.

In [ ]:
import jax

dz_dz_bar = jax.jacobian(lambda z_bar: descent.solve(params={"z_bar": z_bar}).z)(
    jnp.array([1.0, -0.5])
)
dz_dz_bar

Next, [`08_planners`](08_planners.ipynb): planners solve a problem into a policy, and their
compiled face is a parametric program like the one above.